# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [ ]:
aviation_df = pd.read_csv('data/AviationData.csv', encoding='latin-1')
aviation_df.info()
aviation_df.describe()

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [6]:
relevant_cols = [
    'Event.Date', 'Make', 'Model', 'Amateur.Built', 'Aircraft.Category',
    'Number.of.Engines', 'Aircraft.damage', 'Total.Fatal.Injuries',
    'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured',
    'Purpose.of.flight', 'Weather.Condition', 'Broad.phase.of.flight', 'Engine.Type'
]

# Create a clean working copy
aviationrev_df = aviation_df[relevant_cols].copy()
aviationrev_df.dropna(subset=['Make', 'Model', 'Amateur.Built'], inplace=True)
injury_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries',
               'Total.Minor.Injuries', 'Total.Uninjured']
aviationrev_df[injury_cols] = aviationrev_df[injury_cols].fillna(0)

# Filter for professional builds only
aviationrev_df = aviationrev_df[aviationrev_df['Amateur.Built'] == 'No']

# Impute missing Aircraft.Category and then filter for 'Airplane' only as per client's interest
aviationrev_df['Aircraft.Category'] = aviationrev_df['Aircraft.Category'].fillna('Airplane')
aviationrev_df = aviationrev_df[aviationrev_df['Aircraft.Category'] == 'Airplane']

aviationrev_df['Number.of.Engines'] = aviationrev_df['Number.of.Engines'].fillna(aviationrev_df['Number.of.Engines'].mode()[0])
aviationrev_df['Event.Date'] = pd.to_datetime(aviationrev_df['Event.Date'])
aviationrev_df = aviationrev_df[aviationrev_df['Event.Date'].dt.year >= 1983]


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [ ]:
# List of injury columns
injury_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured']

# Impute 0 for all missing injury counts to allow for summation
aviationrev_df[injury_cols] = aviationrev_df[injury_cols].fillna(0)

# Derived Column: Total.Occupants
aviationrev_df['Total.Occupants'] = aviationrev_df[injury_cols].sum(axis=1)
# Derived Column: Severe.Injury.Rate
# Logic: (Fatal + Serious) / Total Occupants
# We use a condition to avoid division by zero errors for empty aircraft.
aviationrev_df['Severe.Injury.Rate'] = 0.0
aviationrev_df.loc[aviationrev_df['Total.Occupants'] > 0, 'Severe.Injury.Rate'] = (
    (aviationrev_df['Total.Fatal.Injuries'] + aviationrev_df['Total.Serious.Injuries']) / aviationrev_df['Total.Occupants']
)
aviationrev_df.tail()

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [ ]:
# Impute 'Unknown' for missing damage status in the working DataFrame
aviationrev_df['Aircraft.damage'] = aviationrev_df['Aircraft.damage'].fillna('Unknown')

# Derived Column: Is.Destroyed
# Logic: If damage is 'DESTROYED', then 'Yes', else 'No'
aviationrev_df['Is.Destroyed'] = aviationrev_df['Aircraft.damage'].apply(lambda x: 'Yes' if x == 'DESTROYED' else 'No')

# Display the head of the DataFrame to verify the new column
aviationrev_df.head()

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

 Cleaning Tasks for 'Make' Column:
1. Standardize Case and Trim Whitespace: Convert all entries to uppercase and remove leading/trailing whitespace.
2. Filter by Frequency: Keep only Makes that appear a reasonable number of times (threshold set to 50 or more occurrences).


In [ ]:
# Execute cleaning tasks
# 1. Convert to uppercase and trim whitespace
aviationrev_df['Make'] = aviationrev_df['Make'].str.upper().str.strip()

# 2. Filter by frequency
make_counts = aviationrev_df['Make'].value_counts()
makes_to_keep = make_counts[make_counts >= 50].index

aviationrev_df = aviationrev_df[aviationrev_df['Make'].isin(makes_to_keep)].copy()
aviationrev_df['Make'].value_counts().head(20)

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [ ]:
# Inspect the 'Model' column for NaNs
aviationrev_df['Model'].isna().sum()

# Drop rows where 'Model' is NaN, as a model is crucial for identifying an aircraft type.
aviationrev_df.dropna(subset=['Model'], inplace=True)
# Inspect if model labels are unique to each make
# Group by 'Make' and count unique 'Model' values for each 'Make'
model_uniqueness = aviationrev_df.groupby('Make')['Model'].nunique()
# Display Makes where a model might not be unique (i.e., multiple Makes have the same model name)
# This is more about if a 'Model' like '172' exists across different 'Makes'.
# Let's check for Models that appear with more than one unique Make.
model_make_counts = aviationrev_df.groupby('Model')['Make'].nunique()
non_unique_models = model_make_counts[model_make_counts > 1]

if not non_unique_models.empty:
    print(non_unique_models.head(10))
else:
    print("No models found that appear under more than one 'Make' in the filtered dataset.")

# Create a derived column that is a unique identifier for a given plane type: 'Make_Model'
# This combines 'Make' and 'Model' to ensure a unique identifier for each aircraft type.
aviationrev_df['Make_Model'] = aviationrev_df['Make'] + ' ' + aviationrev_df['Model']

print(aviationrev_df[['Make', 'Model', 'Make_Model']].head())

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [ ]:
categorical_cols = ['Aircraft.damage', 'Purpose.of.flight', 'Weather.Condition', 'Broad.phase.of.flight','Engine.Type']
aviationrev_df[categorical_cols] = aviationrev_df[categorical_cols].fillna('Unknown')
aviationrev_df.isna().sum()

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [ ]:
# Inspect NaN counts across all columns after previous cleaning steps
aviationrev_df.isna().sum()

# Based on the output, there are currently no columns with a significant number of NaNs
# that would warrant dropping them at this stage, as most have been imputed or filtered.
# If there were columns with a high percentage of NaNs (e.g., >50%), they would be dropped here.
# For example, if 'Example_Column' had many NaNs, you would use:
# if aviationrev_df['Example_Column'].isnull().sum() / len(aviationrev_df) > 0.5:
#     aviationrev_df.drop('Example_Column', axis=1, inplace=True)

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [ ]:
aviationrev_df.to_csv('cleaned_aviation_data.csv', index=False)
